# Feature Engineering Revision Notebook

This notebook covers the main feature engineering steps in simple, revisable code:

1. Encoding categorical variables
2. Feature transformation
3. Feature creation
4. Feature selection

## 1. Setup

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv("Cleaned_Tatanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,S


In [3]:
# Basic data checking
print("Rows and columns:", df.shape)
print("\nColumns:")
print(df.columns)
print("\nMissing values:")
print(df.isnull().sum())

Rows and columns: (418, 11)

Columns:
Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Embarked'],
      dtype='object')

Missing values:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


## 2. Work on a Copy

Always keep the original data safe. Do feature engineering on a copy.

In [4]:
data = df.copy()
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,S


## 3. Basic Missing Value Handling

Feature engineering works best when important columns do not have missing values.

In [5]:
# Fill numeric missing values with median
data["Age"] = data["Age"].fillna(data["Age"].median())
data["Fare"] = data["Fare"].fillna(data["Fare"].median())

# Fill categorical missing values with mode
data["Embarked"] = data["Embarked"].fillna(data["Embarked"].mode()[0])

data.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

# Part A: Encoding Categorical Variables

Machine learning models understand numbers, not text. Encoding means converting categorical/text columns into numeric columns.

## 4. Binary Encoding

Use binary encoding when a column has only two categories.

In [6]:
# Sex has two values: male and female
data["Sex_encoded"] = data["Sex"].map({"male": 0, "female": 1})

data[["Sex", "Sex_encoded"]].head()

,Sex,Sex_encoded
0,male,0
1,female,1
2,male,0
3,male,0
4,female,1


## 5. One-Hot Encoding

Use one-hot encoding when a column has categories with no natural order.

In [7]:
# Embarked has categories like C, Q, S
embarked_dummies = pd.get_dummies(data["Embarked"], prefix="Embarked", dtype=int)

# Add dummy columns to the main data
data = pd.concat([data, embarked_dummies], axis=1)

data[["Embarked", "Embarked_C", "Embarked_Q", "Embarked_S"]].head()

,Embarked,Embarked_C,Embarked_Q,Embarked_S
0,Q,0,1,0
1,S,0,0,1
2,Q,0,1,0
3,S,0,0,1
4,S,0,0,1


## 6. Ordinal Encoding

Use ordinal encoding when categories have a meaningful order. In this dataset, `Pclass` is already ordinal: 1st class, 2nd class, 3rd class.

In [8]:
# Smaller number means higher class, so we can also create a simple rank feature.
data["Class_rank"] = data["Pclass"].map({1: 3, 2: 2, 3: 1})

data[["Pclass", "Class_rank"]].head()

,Pclass,Class_rank
0,3,1
1,3,1
2,2,2
3,3,1
4,3,1


# Part B: Feature Transformation

Feature transformation changes the scale or shape of numeric columns. It can make model training easier.

## 7. Standardization

Standardization changes data to mean 0 and standard deviation 1. It is useful for models like Logistic Regression, KNN, and SVM.

In [9]:
standard_scaler = StandardScaler()

data[["Age_standard", "Fare_standard"]] = standard_scaler.fit_transform(
    data[["Age", "Fare"]]
)

data[["Age", "Age_standard", "Fare", "Fare_standard"]].head()

,Age,Age_standard,Fare,Fare_standard
0,34.5,0.386231,7.8292,-0.497413
1,47.0,1.371370,7.0000,-0.512278
2,62.0,2.553537,9.6875,-0.464100
3,27.0,-0.204852,8.6625,-0.482475
4,22.0,-0.598908,12.2875,-0.417492


## 8. Normalization / Min-Max Scaling

Min-Max scaling changes values to a fixed range, usually 0 to 1.

In [10]:
minmax_scaler = MinMaxScaler()

data[["Age_minmax", "Fare_minmax"]] = minmax_scaler.fit_transform(
    data[["Age", "Fare"]]
)

data[["Age", "Age_minmax", "Fare", "Fare_minmax"]].head()

,Age,Age_minmax,Fare,Fare_minmax
0,34.5,0.452723,7.8292,0.015282
1,47.0,0.617566,7.0000,0.013663
2,62.0,0.815377,9.6875,0.018909
3,27.0,0.353818,8.6625,0.016908
4,22.0,0.287881,12.2875,0.023984


## 9. Log Transformation

Use log transformation when a numeric column is highly skewed. `np.log1p()` is safe for zero values.

In [11]:
data["Fare_log"] = np.log1p(data["Fare"])

data[["Fare", "Fare_log"]].head()

,Fare,Fare_log
0,7.8292,2.178064
1,7.0000,2.079442
2,9.6875,2.369075
3,8.6625,2.268252
4,12.2875,2.586824


## 10. Binning

Binning converts a numeric column into groups. Example: Age to Child, Teen, Adult, Senior.

In [12]:
data["Age_group"] = pd.cut(
    data["Age"],
    bins=[0, 12, 18, 60, 100],
    labels=["Child", "Teen", "Adult", "Senior"]
)

data[["Age", "Age_group"]].head(10)

,Age,Age_group
0,34.5,Adult
1,47.0,Adult
2,62.0,Senior
3,27.0,Adult
4,22.0,Adult
5,14.0,Teen
6,30.0,Adult
7,26.0,Adult
8,18.0,Teen
9,21.0,Adult


# Part C: Feature Creation

Feature creation means making new useful columns from existing columns.

## 11. Family Size Feature

`Family_size = SibSp + Parch + 1`. The `+1` counts the passenger.

In [13]:
data["Family_size"] = data["SibSp"] + data["Parch"] + 1

data[["SibSp", "Parch", "Family_size"]].head()

,SibSp,Parch,Family_size
0,0,0,1
1,1,0,2
2,0,0,1
3,0,0,1
4,1,1,3


## 12. Is Alone Feature

Create a binary feature showing whether the passenger travelled alone.

In [14]:
data["Is_alone"] = (data["Family_size"] == 1).astype(int)

data[["Family_size", "Is_alone"]].head()

,Family_size,Is_alone
0,1,1
1,2,0
2,1,1
3,1,1
4,3,0


## 13. Title Feature from Name

Names contain titles like Mr, Mrs, Miss, Master. We can extract them as a new feature.

In [15]:
data["Title"] = data["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)

# Rare titles are grouped into one category to keep encoding simple.
common_titles = ["Mr", "Mrs", "Miss", "Master"]
data["Title"] = data["Title"].where(data["Title"].isin(common_titles), "Rare")

data[["Name", "Title"]].head()

,Name,Title
0,"Kelly, Mr. James",Mr
1,"Wilkes, Mrs. James (Ellen Needs)",Mrs
2,"Myles, Mr. Thomas Francis",Mr
3,"Wirz, Mr. Albert",Mr
4,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",Mrs


## 14. Encode Created Categorical Feature

After creating a text feature, encode it before model training.

In [16]:
title_dummies = pd.get_dummies(data["Title"], prefix="Title", dtype=int)
data = pd.concat([data, title_dummies], axis=1)

title_dummies.head()

,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,0,1,0,0
1,0,0,0,1,0
2,0,0,1,0,0
3,0,0,1,0,0
4,0,0,0,1,0


## 15. Fare Per Person Feature

If a ticket fare is shared by family members, fare per person can be more useful than total fare.

In [17]:
data["Fare_per_person"] = data["Fare"] / data["Family_size"]

data[["Fare", "Family_size", "Fare_per_person"]].head()

,Fare,Family_size,Fare_per_person
0,7.8292,1,7.829200
1,7.0000,2,3.500000
2,9.6875,1,9.687500
3,8.6625,1,8.662500
4,12.2875,3,4.095833


# Part D: Feature Selection

Feature selection means choosing useful columns and removing unnecessary columns.

## 16. Prepare Final Input and Output

Drop columns that are IDs, raw text, target column, or already replaced by engineered features.

In [18]:
drop_columns = [
    "PassengerId", "Survived", "Name", "Ticket",
    "Sex", "Embarked", "Title", "Age_group"
]

X = data.drop(columns=drop_columns)
y = data["Survived"]

print("Input shape:", X.shape)
print("Output shape:", y.shape)
X.head()

Input shape: (418, 23)
Output shape: (418,)


,Pclass,Age,SibSp,Parch,Fare,Sex_encoded,Embarked_C,Embarked_Q,Embarked_S,Class_rank,...,Fare_minmax,Fare_log,Family_size,Is_alone,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Fare_per_person
0,3,34.5,0,0,7.8292,0,0,1,0,1,...,0.015282,2.178064,1,1,0,0,1,0,0,7.829200
1,3,47.0,1,0,7.0000,1,0,0,1,1,...,0.013663,2.079442,2,0,0,0,0,1,0,3.500000
2,2,62.0,0,0,9.6875,0,0,1,0,2,...,0.018909,2.369075,1,1,0,0,1,0,0,9.687500
3,3,27.0,0,0,8.6625,0,0,0,1,1,...,0.016908,2.268252,1,1,0,0,1,0,0,8.662500
4,3,22.0,1,1,12.2875,1,0,0,1,1,...,0.023984,2.586824,3,0,0,0,0,1,0,4.095833


## 17. Check All Features are Numeric

Most machine learning models need all input columns to be numeric.

In [19]:
X.dtypes

Pclass               int64
Age                float64
SibSp                int64
Parch                int64
Fare               float64
Sex_encoded          int64
Embarked_C           int64
Embarked_Q           int64
Embarked_S           int64
Class_rank           int64
Age_standard       float64
Fare_standard      float64
Age_minmax         float64
Fare_minmax        float64
Fare_log           float64
Family_size          int64
Is_alone             int64
Title_Master         int64
Title_Miss           int64
Title_Mr             int64
Title_Mrs            int64
Title_Rare           int64
Fare_per_person    float64
dtype: object

## 18. Selection by Correlation

Correlation helps us see which numeric features are related to the target.

In [20]:
correlation_with_target = X.corrwith(y).abs().sort_values(ascending=False)

correlation_with_target

Sex_encoded        1.000000
Title_Mr           0.877762
Title_Miss         0.633617
Title_Mrs          0.603458
Is_alone           0.244187
Fare_log           0.222504
Fare               0.192036
Fare_minmax        0.192036
Fare_standard      0.192036
Title_Master       0.173858
Fare_per_person    0.164980
Family_size        0.161803
Parch              0.159120
Embarked_Q         0.115574
Pclass             0.108615
Class_rank         0.108615
Embarked_S         0.105883
SibSp              0.099943
Embarked_C         0.033684
Title_Rare         0.021140
Age_standard       0.008035
Age_minmax         0.008035
Age                0.008035
dtype: float64

## 19. Remove Low Variance Features

A feature with almost the same value in every row usually does not help much.

In [21]:
variance_selector = VarianceThreshold(threshold=0.01)
variance_selector.fit(X)

selected_by_variance = X.columns[variance_selector.get_support()]
removed_by_variance = X.columns[~variance_selector.get_support()]

print("Selected features:", list(selected_by_variance))
print("Removed features:", list(removed_by_variance))

Selected features: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_encoded', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Class_rank', 'Age_standard', 'Fare_standard', 'Age_minmax', 'Fare_minmax', 'Fare_log', 'Family_size', 'Is_alone', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'Fare_per_person']
Removed features: []


## 20. SelectKBest

`SelectKBest` selects the top k features based on a scoring function.

In [22]:
k = min(8, X.shape[1])

kbest_selector = SelectKBest(score_func=f_classif, k=k)
kbest_selector.fit(X, y)

kbest_scores = pd.DataFrame({
    "Feature": X.columns,
    "Score": kbest_selector.scores_
}).sort_values(by="Score", ascending=False)

kbest_scores

c:\Users\RUPESH\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw


,Feature,Score
5,Sex_encoded,inf
19,Title_Mr,1396.363636
18,Title_Miss,279.036855
20,Title_Mrs,238.254545
16,Is_alone,26.377807
14,Fare_log,21.668063
13,Fare_minmax,15.928584
11,Fare_standard,15.928584
4,Fare,15.928584
17,Title_Master,12.966234


## 21. Feature Importance using Random Forest

Tree-based models can tell us which features were most useful for splitting the data.

In [23]:
forest = RandomForestClassifier(random_state=42)
forest.fit(X, y)

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": forest.feature_importances_
}).sort_values(by="Importance", ascending=False)

importance_df

,Feature,Importance
5,Sex_encoded,0.401310
19,Title_Mr,0.251889
18,Title_Miss,0.114219
20,Title_Mrs,0.103442
17,Title_Master,0.016250
10,Age_standard,0.011818
13,Fare_minmax,0.011803
12,Age_minmax,0.010938
11,Fare_standard,0.010493
14,Fare_log,0.009697


## 22. Create Final Selected Feature Set

For revision, choose top features from Random Forest importance. In real projects, compare multiple methods.

In [24]:
top_features = importance_df.head(8)["Feature"].tolist()

X_selected = X[top_features]

print("Selected columns:", top_features)
X_selected.head()

Selected columns: ['Sex_encoded', 'Title_Mr', 'Title_Miss', 'Title_Mrs', 'Title_Master', 'Age_standard', 'Fare_minmax', 'Age_minmax']


,Sex_encoded,Title_Mr,Title_Miss,Title_Mrs,Title_Master,Age_standard,Fare_minmax,Age_minmax
0,0,1,0,0,0,0.386231,0.015282,0.452723
1,1,0,0,1,0,1.371370,0.013663,0.617566
2,0,1,0,0,0,2.553537,0.018909,0.815377
3,0,1,0,0,0,-0.204852,0.016908,0.353818
4,1,0,0,1,0,-0.598908,0.023984,0.287881


## 23. Quick Model Check

This is not the main topic, but it confirms that the selected features can be used in a model.

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 1.0


# Quick Revision Cheat Sheet

| Step | Method | When to Use |
|---|---|---|
| Binary Encoding | `map()` | Two categories like male/female |
| One-Hot Encoding | `pd.get_dummies()` | Categories with no order |
| Ordinal Encoding | `map()` | Categories with order |
| Standardization | `StandardScaler()` | Mean 0, standard deviation 1 |
| Normalization | `MinMaxScaler()` | Scale values between 0 and 1 |
| Log Transform | `np.log1p()` | Reduce skewness |
| Binning | `pd.cut()` | Convert numeric values into groups |
| Feature Creation | New columns | Combine or extract useful information |
| Correlation Selection | `.corrwith()` | Check relation with target |
| Low Variance Removal | `VarianceThreshold()` | Remove almost constant columns |
| SelectKBest | `SelectKBest()` | Select top scoring features |
| Feature Importance | Random Forest | Rank features by model importance |

## Final Revision Flow

1. Load data
2. Handle missing values
3. Encode categorical columns
4. Transform numeric columns
5. Create new useful features
6. Drop unnecessary columns
7. Select best features
8. Train model using selected features

# Model Training

Model training means giving the prepared input features `X` and target column `y` to a machine learning algorithm so it can learn patterns and make predictions.

After feature engineering, the common flow is:

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = ModelName()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
```

There are three main groups of models: regression, classification, and unsupervised learning.

## Regression Models

Regression models are used when the target column is continuous numeric data, such as price, age, salary, marks, sales, or temperature.

| Model | Importing | Syntax | Explanation |
|---|---|---|---|
| Linear Regression | `from sklearn.linear_model import LinearRegression` | `model = LinearRegression()` | Finds a straight-line relationship between features and target. |
| Ridge Regression | `from sklearn.linear_model import Ridge` | `model = Ridge(alpha=1.0)` | Linear regression with L2 regularization to reduce overfitting. |
| Lasso Regression | `from sklearn.linear_model import Lasso` | `model = Lasso(alpha=0.1)` | Linear regression with L1 regularization; can make weak feature coefficients zero. |
| ElasticNet Regression | `from sklearn.linear_model import ElasticNet` | `model = ElasticNet(alpha=0.1, l1_ratio=0.5)` | Combines Ridge and Lasso regularization. |
| Polynomial Regression | `from sklearn.preprocessing import PolynomialFeatures` and `from sklearn.linear_model import LinearRegression` | `poly = PolynomialFeatures(degree=2)` then `model = LinearRegression()` | Creates polynomial features to learn curved relationships. |
| Decision Tree Regressor | `from sklearn.tree import DecisionTreeRegressor` | `model = DecisionTreeRegressor(random_state=42)` | Splits data into decision rules for numeric prediction. |
| Random Forest Regressor | `from sklearn.ensemble import RandomForestRegressor` | `model = RandomForestRegressor(random_state=42)` | Uses many decision trees and averages their predictions. |
| Extra Trees Regressor | `from sklearn.ensemble import ExtraTreesRegressor` | `model = ExtraTreesRegressor(random_state=42)` | Similar to random forest, but uses more random splits. |
| Gradient Boosting Regressor | `from sklearn.ensemble import GradientBoostingRegressor` | `model = GradientBoostingRegressor(random_state=42)` | Builds trees one by one, each correcting previous errors. |
| AdaBoost Regressor | `from sklearn.ensemble import AdaBoostRegressor` | `model = AdaBoostRegressor(random_state=42)` | Combines weak learners and gives more focus to wrong predictions. |
| Support Vector Regressor | `from sklearn.svm import SVR` | `model = SVR(kernel="rbf")` | Uses support vectors to predict continuous values. |
| KNN Regressor | `from sklearn.neighbors import KNeighborsRegressor` | `model = KNeighborsRegressor(n_neighbors=5)` | Predicts using the average target value of nearest data points. |

Basic regression evaluation:

```python
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
```

## Classification Models

Classification models are used when the target column contains categories or labels, such as survived/not survived, spam/not spam, disease/no disease, or class A/class B/class C.

| Model | Importing | Syntax | Explanation |
|---|---|---|---|
| Logistic Regression | `from sklearn.linear_model import LogisticRegression` | `model = LogisticRegression(max_iter=1000)` | Simple and strong baseline model for binary or multiclass classification. |
| KNN Classifier | `from sklearn.neighbors import KNeighborsClassifier` | `model = KNeighborsClassifier(n_neighbors=5)` | Classifies using the majority class of nearest data points. |
| Support Vector Classifier | `from sklearn.svm import SVC` | `model = SVC(kernel="rbf")` | Finds a boundary that separates classes. |
| Decision Tree Classifier | `from sklearn.tree import DecisionTreeClassifier` | `model = DecisionTreeClassifier(random_state=42)` | Creates decision rules for class prediction. |
| Random Forest Classifier | `from sklearn.ensemble import RandomForestClassifier` | `model = RandomForestClassifier(random_state=42)` | Uses many decision trees and predicts by majority voting. |
| Extra Trees Classifier | `from sklearn.ensemble import ExtraTreesClassifier` | `model = ExtraTreesClassifier(random_state=42)` | Similar to random forest, with more randomness in tree splits. |
| Gradient Boosting Classifier | `from sklearn.ensemble import GradientBoostingClassifier` | `model = GradientBoostingClassifier(random_state=42)` | Builds models step by step to fix previous classification mistakes. |
| AdaBoost Classifier | `from sklearn.ensemble import AdaBoostClassifier` | `model = AdaBoostClassifier(random_state=42)` | Combines weak classifiers into a stronger classifier. |
| Gaussian Naive Bayes | `from sklearn.naive_bayes import GaussianNB` | `model = GaussianNB()` | Works well with continuous numeric features that follow a normal-like distribution. |
| Multinomial Naive Bayes | `from sklearn.naive_bayes import MultinomialNB` | `model = MultinomialNB()` | Commonly used for count-based text features. |
| Bernoulli Naive Bayes | `from sklearn.naive_bayes import BernoulliNB` | `model = BernoulliNB()` | Useful for binary features like word present/not present. |

Basic classification evaluation:

```python
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

accuracy = accuracy_score(y_test, y_pred)
matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)
```

## Unsupervised Learning Models

Unsupervised learning is used when there is no target column `y`. The model learns patterns, groups, hidden structure, or unusual points from only input features `X`.

| Model | Importing | Syntax | Explanation |
|---|---|---|---|
| K-Means Clustering | `from sklearn.cluster import KMeans` | `model = KMeans(n_clusters=3, random_state=42)` | Divides data into `k` groups based on similarity. |
| Agglomerative Clustering | `from sklearn.cluster import AgglomerativeClustering` | `model = AgglomerativeClustering(n_clusters=3)` | Hierarchical clustering that merges similar points step by step. |
| DBSCAN | `from sklearn.cluster import DBSCAN` | `model = DBSCAN(eps=0.5, min_samples=5)` | Finds dense groups and can detect noise/outliers. |
| Gaussian Mixture Model | `from sklearn.mixture import GaussianMixture` | `model = GaussianMixture(n_components=3, random_state=42)` | Soft clustering model where each point gets probability of belonging to clusters. |
| PCA | `from sklearn.decomposition import PCA` | `model = PCA(n_components=2)` | Reduces many features into fewer important components. |
| Truncated SVD | `from sklearn.decomposition import TruncatedSVD` | `model = TruncatedSVD(n_components=2)` | Dimensionality reduction, often useful for sparse text data. |
| t-SNE | `from sklearn.manifold import TSNE` | `model = TSNE(n_components=2, random_state=42)` | Reduces dimensions mainly for visualization. |
| Isolation Forest | `from sklearn.ensemble import IsolationForest` | `model = IsolationForest(random_state=42)` | Detects unusual or abnormal data points. |
| One-Class SVM | `from sklearn.svm import OneClassSVM` | `model = OneClassSVM(kernel="rbf")` | Learns the normal pattern and identifies outliers. |

Basic unsupervised syntax:

```python
# Clustering models usually use fit_predict
labels = model.fit_predict(X)

# Dimensionality reduction models usually use fit_transform
X_reduced = model.fit_transform(X)

# GaussianMixture uses fit and predict
model.fit(X)
labels = model.predict(X)
```

## Quick Model Selection Guide

| Problem Type | Target Column | Example | Models to Try First |
|---|---|---|---|
| Regression | Continuous numeric value | Predict house price | `LinearRegression`, `RandomForestRegressor`, `GradientBoostingRegressor` |
| Classification | Category or class label | Predict survived/not survived | `LogisticRegression`, `RandomForestClassifier`, `SVC` |
| Clustering | No target column | Group customers | `KMeans`, `AgglomerativeClustering`, `DBSCAN` |
| Dimensionality Reduction | No target column | Reduce features for visualization | `PCA`, `TruncatedSVD`, `TSNE` |
| Anomaly Detection | No target column or rare abnormal labels | Detect fraud/outliers | `IsolationForest`, `OneClassSVM` |

Important note: use regression models for numeric prediction, classification models for label prediction, and unsupervised models when the target column is not available.